# Build Dimension Customer
1. read the silver customers table
2. create the customer surrogate key
3. select the required columns
4. write the transformed data to gold dim_customer table

In [0]:
#Imports
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

### Step1 - read the silver customers table

In [0]:
customers_df = spark.read.table("olist_catalog.silver.customers")

### Step2 - create the customer surrogate key and select the required columns

In [0]:
window_spec = Window.orderBy('customer_id')
dim_customer_df = (
    customers_df.withColumn('customer_sk', row_number().over(window_spec))
        .select("customer_sk","customer_id","customer_unique_id","customer_city","customer_state")
    )

### Step3 - write the transformed data to gold dim_customer table

In [0]:
(
    dim_customer_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("olist_catalog.gold.dim_customers")
    )

In [0]:
%sql
select * from olist_catalog.gold.dim_customers